In [1]:
import torch
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image
import os
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained ResNet
model = models.resnet50(pretrained=True)
model.fc = torch.nn.Identity()
model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],
                         [0.229,0.224,0.225])
])

def extract_feature(img_path):
    img = Image.open(img_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        feature = model(img)
    return feature.cpu().numpy().flatten()


d:\python_ws\pytorch\torch-env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\python_ws\pytorch\torch-env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [2]:

X = []
y = []
class_map = {}

for idx, class_name in enumerate(os.listdir("../dataset/train")):
    class_map[class_name] = idx
    class_dir = os.path.join("../dataset/train", class_name)
    print()

    for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_name)
        X.append(extract_feature(img_path))
        y.append(idx)

X = np.array(X)
y = np.array(y)
print(X.shape)







(6000, 2048)


In [3]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y
)

svm = SVC(kernel="rbf", C=10, gamma="scale")
svm.fit(X_train, y_train)

SVC(C=10)

In [4]:
from sklearn.metrics import classification_report

y_pred = svm.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00       200
           1       0.98      0.96      0.97       200
           2       0.99      0.98      0.98       200
           3       1.00      1.00      1.00       200
           4       0.95      0.99      0.97       200
           5       1.00      0.98      0.99       200

    accuracy                           0.99      1200
   macro avg       0.99      0.99      0.99      1200
weighted avg       0.99      0.99      0.99      1200



In [5]:
import joblib

joblib.dump(svm, "svm_model.pkl")
joblib.dump(class_map, "class_map.pkl")

['class_map.pkl']

In [7]:
import os
lis=os.listdir("../test_data")
for li in lis:
    feature = extract_feature("../test_data/"+li)
    prediction = svm.predict([feature])
    # print("Predicted class:", list(class_map.keys())[prediction[0]])

    if list(class_map.keys())[prediction[0]]:
        print(li, list(class_map.keys())[prediction[0]])

300002_11_C.png C
300002_12_C.png C
300002_13_C.png C
300002_15_A.png A
300002_16_B.png B
300002_17_C.png C
300002_19_A.png A
300002_20_C.png C
300002_21_A.png A
300002_22_A.png A
300002_24_B.png B
300002_25_C.png C
300002_26_B.png B
300002_27_A.png A
300002_2_A.png A
300002_30_A.png A
300002_31_A.png A
300002_32_A.png A
300002_33_A.png A
300002_34_A.png A
300002_35_A.png astric
300002_37_B.png B
300002_3_A.png A
300002_41_B.png B
300002_46_C.png C
300002_47_C.png C
300002_50_B.png B
300002_7_A.png A
300002_9_A.png A
300038_11_C.png C
300038_12_C.png C
300038_15_B.png B
300038_18_B.png B
300038_1_C.png C
300038_20_B.png D
300038_22_B.png C
300038_23_B.png B
300038_24_B.png B
300038_25_B.png C
300038_26_B.png B
300038_28_B.png B
300038_29_B.png B
300038_35_C.png C
300038_37_B.png B
300038_42_D.png D
300038_8_B.png B
300040_12_C.png C
300040_26_D.png D
300040_2_C.png C
300040_35_C.png C
300040_36_D.png D
300050_11_D.png D
300050_40_D.png D
300050_41_D.png D
300051_11_D.png D
300051_36_D.